In [ ]:
# %pip install pillow

Note: you may need to restart the kernel to use updated packages.


In [21]:
import base64
import os

def generate_base64_string(size):
    """Generates a random base64 encoded string of the given size."""
    random_bytes = os.urandom(size)
    base64_string = base64.b64encode(random_bytes).decode('utf-8')
    return base64_string

# Generate a base64 encoded string of 5000 bytes
large_text = generate_base64_string(5000)
print(large_text)

trdpBsKCrxk1VRMskeo8yh7s4AhigiovZ80uTh1rUgJlcKz2keqT36gYKi+DqUTb0yXp1nI8VpynS0fQvq3pUiGM7rTdBbXVhokpmXLLBwBOWQw8RRmKWpmEpEPya+nfnRnGkYXJreTbilan6JSTSVexiB5sUkC9pQMOg0/sR+8FFtD9MaH846v09l/SMaR/KC9ID6/NgoJBaftgPuqAIo66/2mkhG3HDEm/zC3y8Owb46tB9XUB9SMAuFgFUNBq5etgRLablEa5Z1JlJ87CKkuvUQ5wQGTbC/2P7pEiKwfrlst1YxlvnrM36ras+SdWazDMMB5LEIU5xAKlE5BbMKuu7UARWDCQZzBe2cBXHDxklhbEhFseOdSA8gwnNABotcU2Bqrki50GrFML89Bb2B1YG5qqXq9pGGYdML5Gl0275WhHp+7+C0llsQU8TwNdAPnrLAWFNYXDeAK3XPAGSdbv3vYIfdA/J6F4Q4zodUnUhIMKxCXyjf08+8n02S/Fg6k5co2Rs3ASqIfz6x+9zjQf9IM4ROTts7Fc5CKVvwA1humRbMe4XG3vXSarSKz9rLaFXZDL53AsgJ1XL//F3apuGSSbMmYKiTHP1V1UDNa92VFIwTAF0eJUErrp7lTaoNHwpO6UBzAQXaE9H4nkJ1vnsuO0lO3U5uVT/LAqDWlg8f4d4VLNY4R2jShPf6j9UthdpDcSKiSfhX6hR9Fl0DEmLS1K+1ecRWOoI2AZLqal/EJKh7wm9n5ZCdQIx0jVQTVkhj0wgVE/UQHc24MjCvdiWsAnkyPK6LsK45/1jPRz/K8g/JvE2yLttDSRqdnu1lMnZObzYqYzU7UFW0/9KLV43Hlv18geooftrcZR51ehhj1YQWroUns8fcwIylwmnSF3PWOnuQXiJ2I8FbLibxQqdQF3MYDvx4TturGLVqGY1E8Ajw9+4zOiAbeF/wXUCaEgpG3eaBQ0ElQ11p56WCLHjzCBm4O5BBYJZ5lV

In [22]:
import qrcode
from PIL import Image 
from datetime import datetime

In [26]:
import shutil

# Delete previously generated similar type folders
for folder in os.listdir('.'):
    if folder.startswith('gen_output_temp_') and os.path.isdir(folder):
        shutil.rmtree(folder)

# Create a new directory with a unique name based on the current timestamp
gen_output_temp_dir = datetime.now().strftime("gen_output_temp_%Y%m%d_%H%M%S")
os.makedirs(gen_output_temp_dir, exist_ok=True)

In [42]:
# @title Generate a QR code from that large text

def split_text(text, max_length):
    """Splits the text into chunks of max_length."""
    return [text[i:i + max_length] for i in range(0, len(text), max_length)]


def text_to_qrcodes(text, max_length=2953):
    """Converts a large text into multiple QR codes if necessary."""
    chunks = split_text(text, max_length)
    for i, chunk in enumerate(chunks):
        image_name = f'qrcode_{i + 1}.png'
        generate_qr_code(chunk, image_name)


def generate_qr_code(data, image_name):
    """Generates a QR code from the given data and saves it as an image."""
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=10,
        border=4,
    )
    qr.add_data(data)
    qr.make(fit=True)

    img = qr.make_image(fill='black', back_color='white').convert('L')
    img = img.resize((700, 700), Image.Resampling.LANCZOS)
    img.save(os.path.join(gen_output_temp_dir, image_name))



# Example usage
if __name__ == "__main__":
    text_to_qrcodes(large_text)   

In [30]:
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [41]:
import cv2
import os


def images_to_video(image_folder, output_video, fps=1):
    """Creates a video file from multiple images."""
    images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
    images.sort()  # Ensure the images are in the correct order

    # Read the first image to get the dimensions
    frame = cv2.imread(os.path.join(image_folder, images[0]))
    height, width, layers = frame.shape

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 files
    video = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    for image in images:
        img_path = os.path.join(image_folder, image)
        frame = cv2.imread(img_path)
        video.write(frame)

    video.release()

# Example usage
if __name__ == "__main__":
    last_output_dir = gen_output_temp_dir
    images_to_video(last_output_dir, 'output_video.mp4', fps=1)

In [37]:
pip install opencv-python qrcode[pil] pillow pyzbar

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
import os
import qrcode
from PIL import Image
from pyzbar.pyzbar import decode

def extract_images_from_video(video_path, output_folder, resize_dim=(700, 700)):
    """Extracts images from a video file, resizes them, and saves them to the output folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    cap = cv2.VideoCapture(video_path)
    count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Resize the frame to the specified dimensions
        resized_frame = cv2.resize(frame, resize_dim, interpolation=cv2.INTER_AREA)
        img_name = os.path.join(output_folder, f"frame_{count + 1}.png")
        cv2.imwrite(img_name, resized_frame)
        count += 1

    cap.release()


def read_qr_code(image_path):
    """Reads a QR code from an image and returns the decoded text."""
    img = Image.open(image_path)
    decoded_objects = decode(img)
    if decoded_objects:
        return decoded_objects[0].base64.b64encode(random_bytes).decode('utf-8')
    return ""

def extract_text_from_images(image_folder):
    """Extracts and concatenates text from QR codes in images in the specified folder."""
    images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
    images.sort()  # Ensure the images are in the correct order

    full_text = ""
    for image in images:
        img_path = os.path.join(image_folder, image)
        text = read_qr_code(img_path)
        full_text += text

    return full_text

# Example usage
if __name__ == "__main__":
    video_path = 'output_video.mp4'
    output_folder = 'extracted_images'
    
    # Step 1: Extract images from video
    extract_images_from_video(video_path, output_folder)
    
    # Step 2: Extract text from images
    large_text_new = extract_text_from_images(output_folder)
    


AttributeError: 'Decoded' object has no attribute 'decode'

In [32]:
print((len(large_text_new) == len(large_text)))

True


In [40]:
print(large_text_new[1000:])
print(large_text[1000:])

...


In [38]:
large_text_new[1000:] == (large_text[1000:])

False

In [33]:
if large_text_new == large_text:
    print("Success")